# Distortion pedalboard

The distortion section behaves like a small pedalboard. Each pedal
has its own enable bit in `distortion_control.ctrlD`; the section is
gated by the existing `gate_control` bit 2 master flag (the same flag
the legacy `set_guitar_effects(distortion_on=True)` API drives).

**Loud-output warning.** RAT and metal can be very loud at high
drive. Start with `level=30..40` and a quiet monitor. Verify the
default settings (master OFF, no pedal selected, level=35, mix=100)
before enabling anything.

**Bitstream note.** This notebook only makes sense once a build with
non-degraded timing is on the board. If `WNS` was significantly
negative for the deployed bitstream, results will be unstable; do
not run effect-on cells in that case.

Sections:
1. Overlay + ADC HPF state
2. List the pedals and their bit positions
3. Initial cached settings
4. Master ON, then cycle each pedal exclusively
5. Live parameter cell (re-run while playing)
6. Stack pedals (advanced; non-exclusive)
7. Reset to all-off and drop the chain

In [ ]:
from audio_lab_pynq.AudioLabOverlay import AudioLabOverlay

ovl = AudioLabOverlay()
print('ADC HPF state:', ovl.codec.get_adc_hpf_state())
print('R19_ADC_CONTROL:', hex(ovl.codec.R19_ADC_CONTROL[0]))
print('overlay loaded; distortion section starts disabled by default')

## 2. Available pedals

Each entry corresponds to one bit of the pedal-enable mask in
`distortion_control.ctrlD`. Every pedal in the list now has a
working Clash stage in the deployed bitstream; bit 7 is the only
remaining reserved slot, held for a future 8th pedal.

- `clean_boost`     (bit 0, implemented)
- `tube_screamer`   (bit 1, implemented)
- `rat`             (bit 2, mapped onto the existing RAT stage)
- `ds1`             (bit 3, implemented; BOSS DS-1 style)
- `big_muff`        (bit 4, implemented; Big Muff Pi style)
- `fuzz_face`       (bit 5, implemented; Fuzz Face style)
- `metal`           (bit 6, implemented)
- bit 7             reserved (future 8th pedal slot)

In [ ]:
for name in AudioLabOverlay.DISTORTION_PEDALS:
    bit = AudioLabOverlay._DIST_PEDAL_BIT[name]
    impl = name in AudioLabOverlay.DISTORTION_PEDALS_IMPLEMENTED
    flag = 'implemented' if impl else 'reserved'
    print('  bit {:>1}  {:<14} ({})'.format(bit, name, flag))

## 3. Initial cached settings

Defaults come from `AudioLabOverlay.DISTORTION_DEFAULTS`. They are
intentionally quiet (`level=35`, `mix=100`, no pedal selected) so
the overlay never produces a loud transient at load time.

In [ ]:
print(ovl.get_distortion_settings())

## 4. Cycle each pedal (exclusive, safe)

Bring the section master up, then walk through every pedal with
`exclusive=True` so exactly one pedal is active at a time. Use this
with the input shorted or muted on the first try.

In [ ]:
ovl.set_guitar_effects(distortion_on=True)
for name in AudioLabOverlay.DISTORTION_PEDALS:
    ovl.set_distortion_pedal(name, enabled=True, exclusive=True)
    ovl.set_distortion_settings(
        drive=40,
        tone=50,
        level=35,
        bias=50,
        tight=50,
        mix=100,
    )
    settings = ovl.get_distortion_settings()
    print('{:<14} mask=0x{:02x}  drive={}  level={}  pedals={}'.format(
        name,
        settings['pedal_mask'],
        settings['drive'],
        settings['level'],
        {k: v for k, v in settings['pedals'].items() if v},
    ))

## 5. Live parameter cell

Edit any of the values below and re-run the cell to hear the change
without restarting the overlay. Switching pedals via this cell uses
`exclusive=True` to keep the chain to one voicing.

In [ ]:
ovl.set_distortion_settings(
    pedal='tube_screamer',  # try: clean_boost, rat, ds1, big_muff, fuzz_face, metal
    drive=55,
    tone=55,
    level=35,
    bias=50,
    tight=55,
    mix=100,
    exclusive=True,
)
print(ovl.get_distortion_settings())

## 6. Stack pedals (advanced)

`exclusive=False` lets you arm more than one distortion pedal at
once. The chain order in the FPGA is: `clean_boost -> tube_screamer
-> existing RAT (when bit 2 is set) -> metal -> ds1 -> big_muff ->
fuzz_face`. Stacking high-gain pedals (especially metal + big_muff
or fuzz_face on top) can get loud quickly — drop level before
trying.

In [ ]:
ovl.set_distortion_settings(level=25)
ovl.set_distortion_pedals(clean_boost=True, tube_screamer=True, metal=False)
print(ovl.get_distortion_settings())

## 7. All off

Always run this before disconnecting headphones.

In [ ]:
ovl.clear_distortion_pedals()
ovl.set_distortion_settings(drive=20, tone=50, level=35,
                            bias=50, tight=50, mix=100)
ovl.set_guitar_effects(distortion_on=False)
print(ovl.get_distortion_settings())